# Taller Práctico #2 — Sistema RAG para la atención al cliente de EcoMarket

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yeavil-neny/IA_Generativa_Taller2_RAG/blob/main/notebooks/Taller2_RAG_EcoMarket.ipynb)

**Curso:** IA Generativa — Unidad 3: Automatización de Tareas con IA Generativa (Semana 4)
**Caso de estudio:** EcoMarket, e-commerce de productos sostenibles
**Equipo:** Cristian Humberto Yepes Marin y Yenny Alexandra Villarreal Suaza

---

## Objetivo de este notebook

En el **Taller Práctico #1** se construyó un asistente de atención al cliente para EcoMarket que resolvía dos flujos definidos (consultar el estado de un pedido y evaluar una devolución) con plantillas de prompt cuidadosamente diseñadas y un *mini-RAG* didáctico: el código inyectaba directamente el JSON del pedido o la política como contexto.

Este notebook implementa el **Taller Práctico #2**: extender el asistente para que responda **cualquier tipo de solicitud** sobre la empresa mediante un sistema **RAG** (Retrieval Augmented Generation), o comunique con honestidad que no tiene las herramientas para atender la solicitud.

## Cómo está organizado

| Sección | Contenido | Corresponde a |
|---|---|---|
| 1. Preparación del entorno | Clonado del repositorio, dependencias y clave de API | Requisitos de Fase 3 |
| 2. La base de conocimiento | Los 4 tipos de documentos y la segmentación (*chunking*) | Fase 2 |
| 3. Indexación | Embeddings open-source y base vectorial ChromaDB | Fases 1 y 2 |
| 4. Recuperación por similitud | Búsqueda top-k, distancias y umbral de pertinencia | Fase 1 y Fase 3 |
| 5. El LLM y la plantilla de prompts | Herencia del patrón del Taller Práctico #1 | Fase 3 |
| 6. El asistente completo | Función final que integra todas las piezas | Fase 3 |
| 7. Casos de prueba | Cinco casos que cubren la rúbrica | Evidencias Fase 3 |
| 8. Experimento de control | La misma consulta sin RAG (¿qué pasa si no hay contexto?) | Fase 3 (rúbrica: "modificar y observar cambios") |
| 9-11. Limitaciones, ética y conclusiones | Reflexión final | Requerimiento del curso (Semana 4) |

Documentación completa por fase (en formato Markdown, dentro del repositorio):

- `docs/fase1-componentes-rag.md` — selección y justificación de componentes (modelo de embeddings y base de datos vectorial).
- `docs/fase2-base-conocimiento.md` — identificación de documentos, estrategia de segmentación e indexación.
- `docs/fase3-arquitectura-y-limitaciones.md` — arquitectura del flujo, adaptaciones sobre el script de referencia, limitaciones y suposiciones, reflexión sobre escalabilidad y riesgos éticos.


## De dos flujos a cualquier pregunta — y la lectura desde la Semana 4

El enunciado del Taller Práctico #2 pide que el asistente pueda responder "ante cualquier tipo de solicitud, ya sea con la respuesta que parece correcta o haciéndole saber al usuario que no tiene las herramientas para atender su solicitud".

Este comportamiento se logra combinando dos ideas, vistas esta semana en la unidad 3 (Agentes y Agentic Workflows):

1. **Herramientas que amplían el conocimiento del agente.** En la clasificación de herramientas vista en clase, la recuperación de datos es una herramienta de "ampliación de conocimiento". El recuperador RAG juega ese rol: le da al asistente acceso a la base de conocimiento interna de EcoMarket (política de devoluciones, catálogo, FAQ y registro de pedidos), que no está en el conocimiento general del modelo.
2. **Las herramientas definen el alcance, y el agente reflexiona antes de actuar.** Una herramienta bien seleccionada "determina las tareas que el agente puede realizar". Por eso el sistema añade una **verificación de pertinencia**: si la búsqueda por similitud no recupera material suficientemente cercano a la consulta, el asistente no invoca al LLM con contexto pobre (lo que fomentaría alucinaciones, como quedó demostrado en el Taller Práctico #1); en su lugar responde de forma determinista que no dispone de esa información y ofrece escalar a un asesor humano. Esto materializa el proceso de *reflexión* durante la ejecución: monitorear el resultado de la herramienta antes de la siguiente acción e identificar desviaciones tempranamente.


## 1. Preparación del entorno

El notebook está diseñado para ejecutarse en Google Colab. Hace tres cosas:

1. **Clona este repositorio** para acceder, desde una única fuente de verdad, a los archivos de la base de conocimiento (`data/`) y al código del sistema (`src/`).
2. **Instala las dependencias** desde `requirements.txt` del propio repositorio (LangChain y sus integraciones, ChromaDB y Sentence-Transformers). La instalación silenciosa toma un par de minutos solo la primera vez.
3. **Solicita la clave API de Gemini** con `getpass`: se pide por teclado, se guarda solo en la variable de entorno del *runtime* (no queda impresa, no queda en el notebook, no queda en el repositorio). El modelo de embeddings es open-source y no requiere clave.


In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/yeavil-neny/IA_Generativa_Taller2_RAG.git"
RUTA_REPO = pathlib.Path("/content/IA_Generativa_Taller2_RAG")

# Clonar el repositorio si aún no está presente en el runtime (idempotente)
if not RUTA_REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(RUTA_REPO)], check=True)

# Trabajar desde la raíz del repositorio y hacer importable el código de src/
os.chdir(RUTA_REPO)
if str(RUTA_REPO / "src") not in sys.path:
    sys.path.append(str(RUTA_REPO / "src"))

print("Repositorio listo en:", os.getcwd())


In [ ]:
# Dependencias del propio repositorio (ver requirements.txt)
%pip install -q -r requirements.txt


In [ ]:
# Clave API de Gemini: se pide una única vez por runtime y no se muestra ni se registra
import getpass

os.environ.setdefault("GEMINI_API_KEY", "")
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY (pegar, queda oculta): ")
print("Clave configurable registrada (oculta). El modelo de embeddings no requiere clave")


## 2. La base de conocimiento — los cuatro tipos de documentos (Fase 2)

El rendimiento de un sistema RAG empieza por la calidad de lo que puede recuperar. La base de conocimiento del asistente integra cuatro tipos de documentos del negocio, elegidos porque cubren las consultas reales de un canal de soporte de e-commerce:

| Documento | Archivo | Contenido |
|---|---|---|
| Política de devoluciones | `data/politicas_devolucion.md` | Reglas del negocio: plazos, categorías no devolvibles, excepciones y proceso |
| Registro de pedidos | `data/pedidos.json` | 12 pedidos con seguimiento, cliente, productos, estado y fechas |
| Inventario de productos | `data/inventario_productos.csv` | 18 productos con precio, stock y disponibilidad |
| Preguntas frecuentes | `data/faqs.json` | 12 FAQ de envíos, pagos, suscripciones y atención humana |

En el Taller Práctico #1 la política y los pedidos ya se usaban, pero como inyección directa ("mini-RAG didáctico"). Aquí entran a un índice semántico y se les unen dos tipos nuevos (inventario y FAQ) que permiten atender consultas de catálogo y de información general.

### Segmentación por tipo de documento

La decisión central de la Fase 2: **cada tipo de documento se segmenta según su unidad semántica natural**, no con un tamaño fijo arbitrario.

- **La política se divide por secciones** (cada sección `##` del Markdown es una chunk propia): cada sección responde una familia de consultas distinta y cabe completa dentro del límite de entrada del modelo de embeddings (128 tokens), elegido en la Fase 1 — por eso el techo de chunk se calibró MIDENDO el tokenizador del modelo: el corpus entero queda bajo los 128 tokens y no hay truncamientos silenciosos (verificado en la sección 3).
- **Los pedidos se segmentan por registro** (un chunk por pedido): mezclar campos de pedidos distintos en un mismo fragmento induciría al modelo a error.
- **El inventario, por fila** (un chunk por SKU) y **las FAQ, por par pregunta-respuesta**: la pregunta aporta la superficie semántica que más se parece a la consulta real del usuario, y el chunk que la contiene llega al prompt ya con su respuesta dentro.

La comparación completa de estrategias (tamaño fijo / párrafos / recursiva) y su justificación está en `docs/fase2-base-conocimiento.md`, sección 2.2. En este notebook se verifica dos veces: con un mini-experimento de chunking (sección 4.3) y midiendo token a token que ningún chunk excede el límite del modelo (sección 3).


In [ ]:
# Los cargadores viven en src/rag_ecomarket.py (misma función que construye el índice final)
import rag_ecomarket as rag

politica = rag.cargar_politica("data/politicas_devolucion.md")
pedidos = rag.cargar_pedidos("data/pedidos.json")
inventario = rag.cargar_inventario("data/inventario_productos.csv")
faqs = rag.cargar_faqs("data/faqs.json")

print("Chunks generados por tipo de documento:")
print(f"  politica:   {len(politica):3d} chunks (uno por sección Markdown)")
print(f"  pedido:     {len(pedidos):3d} chunks (uno por pedido)")
print(f"  inventario: {len(inventario):3d} chunks (uno por SKU)")
print(f"  faq:        {len(faqs):3d} chunks (uno por pregunta)")
print(f"  TOTAL:      {len(politica) + len(pedidos) + len(inventario) + len(faqs)} chunks")

print("\nEjemplo de chunk de la política (sección completa, semánticamente autónoma):")
print(politica[1].page_content[:400])
print("...")

print("\nEjemplo de chunk de pedido (registro completo ECO-10003):")
print(pedidos[2].page_content)


## 3. Indexación — embeddings y base vectorial (Fase 1 puesta en práctica)

Los componentes seleccionados y justificados en `docs/fase1-componentes-rag.md`:

- **Modelo de embeddings:** `paraphrase-multilingual-MiniLM-L12-v2` (Sentence-Transformers, código abierto). Se eligió frente a las APIs propietarias (OpenAI, Google) porque el corpus es pequeño, el costo debe ser cero, y la vectorización ocurre dentro del entorno del proyecto: ningún texto del negocio (ni nombres de clientes en los pedidos) viaja a un servicio externo solo para calcular embeddings — decisión coherente con la política de privacidad (Ley 1581) argumentada en el Taller Práctico #1. Frente a otros open-source más pesados (por ejemplo, e5-base), MiniLM aporta calidad suficiente en español para un corpus curado, con la fracción del costo de cómputo y la ventaja de correr sin GPU en Colab.
- **Base de datos vectorial:** ChromaDB (código abierto, persistente en disco), elegida frente a Pinecone (servicio gestionado de pago que el volumen no justifica todavía) y frente a Weaviate (capacidades superiores que el caso no necesita, a cambio de cargar la operación de un servicio; la carga MLOps ya se descartó por el mismo motivo en el Taller Práctico #1).
- **Espacio de similitud: coseno.** La distancia devuelve 0 para textos idénticos en dirección y valores crecientes para textos menos similares; es interpretable y permite definir el umbral de pertinencia de la sección 4.

El indexado (carga -> segmentación -> vectorización -> upsert) se reconstruye desde cero en cada corrida: el corpus completo procesa en menos de un segundo, y así se garantiza que el índice siempre refleje los archivos vigentes de `data/`.


In [ ]:
# Construye el índice: carga los 4 tipos de documentos, segmenta, calcula
# embeddings y persiste en chroma_ecomarket/ (espacio coseno)
vectorstore = rag.construir_base_conocimiento()


### 3.1 Verificación: ¿todo chunk cabe en el modelo de embeddings?

El modelo elegido trunca su entrada a 128 tokens (y trunca silenciosamente: no avisa). Esta celda mide con el tokenizador real del modelo los chunks ya indexados, para confirmar que la segmentación de la Fase 2 es compatible con el componente elegido en la Fase 1: la decisión del techo de tamaño no es arbitraria, sino resultado de esa medición.


In [ ]:
# Mide los tokens de TODOS los chunks indexados con el tokenizador real del modelo
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer(rag.MODELO_EMBEDDINGS)
tokenizador = modelo_embeddings.tokenizer

registros = vectorstore.get(include=["documents"])
n_tokens = [len(tokenizador(texto)["input_ids"]) for texto in registros["documents"]]
exceden = sum(1 for n in n_tokens if n > 128)

print(f"Chunks indexados:                {len(n_tokens)}")
print(f"Tokens por chunk (promedio):     {sum(n_tokens) / len(n_tokens):.0f}")
print(f"Tokens por chunk (máximo):       {max(n_tokens)}")
print(f"Límite del modelo de embeddings: 128 tokens")
print(f"Chunks que excederían el límite y se truncarían: {exceden}")


## 4. Recuperación por similitud y umbral de pertinencia

Antes de generar cualquier respuesta, el sistema consulta la base vectorial. Dos decisiones concretas:

- **Búsqueda top-k con distancias en mano.** Se usa `similarity_search_with_score` (y no la cadena genérica `RetrievalQA` del script de referencia) porque el asistente necesita *leer* el resultado de su herramienta antes de actuar: las distancias le permiten decidir si el material recuperado es pertinente.
- **Umbral de pertinencia sobre la distancia coseno.** Si la distancia del mejor chunk excede `UMBRAL_DISTANCIA = 0.55` (es decir, su similitud coseno con la consulta queda por debajo de 0.45), el sistema no consulta al LLM: responde con el mensaje determinista de escalamiento. El umbral se calibró con los casos de prueba de este notebook (las consultas dentro del dominio obtienen distancias claramente menores que las fuera de dominio) y puede ajustarse con `--umbral` o recalibrarse para otro corpus.

Nótese el enrutamiento: si la consulta contiene un número de seguimiento (patrón `ECO-XXXXX`), la búsqueda se filtra a los documentos de tipo `pedido` — los chunks de política o catálogo serían ruido para ese tipo de consulta.


In [ ]:
# Recuperación demostrativa: la consulta "¿puedo devolver el café?" comparte
# poco vocabulario con la política, pero el embedding semántico la acerca al
# pasaje correcto (que dice "alimentos y bebidas... no admiten devolución")
consulta_demo = "¿Puedo devolver el café orgánico que compré?"
resultados = rag.recuperar(vectorstore, consulta_demo, k=4)

print(f"Consulta: {consulta_demo}")
print("Chunks más similares (tipo, distancia coseno — menor es más similar):\n")
for doc, distancia in resultados:
    meta = doc.metadata
    identificador = meta.get("tracking") or meta.get("sku") or meta.get("seccion") or meta.get("id", "")
    print(f"[distancia={distancia:.3f}] {meta.get('tipo')} {identificador}")
    print(doc.page_content[:220], "...")
    print()


### 4.3 Mini-experimento de segmentación: tamaño fijo frente a segmentación por unidad semántica

La Fase 2 sostuvo que un tamaño fijo arbitrario puede cortar reglas por la mitad. Este experimento lo comprueba sobre la política de devoluciones: se segmenta el mismo texto con `RecursiveCharacterTextSplitter` en modo "puro" (tamaño fijo de 512 caracteres, sin considerar las secciones) y se compara con la segmentación por secciones que usa el sistema.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

texto_politica = open("data/politicas_devolucion.md", encoding="utf-8").read()

# Estrategia A: splitter "genérico" sobre el texto crudo, tamaño fijo de 512
# caracteres (la forma más simple, similar a la de rag_ejemplo.py; no sabe
# que el documento tiene estructura de encabezados y listas)
splitter_fijo = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=80)
chunks_fijos = [c for c in splitter_fijo.split_text(texto_politica) if c.strip()]

# Estrategia B (la del sistema): un chunk por sección, cada uno con su etiqueta
chunks_seccion = [c.page_content for c in politica]

def buscar(chunks, termino):
    """Devuelve los chunks que contienen el término buscado."""
    return [ch for ch in chunks if termino in ch.lower()]

termino = "liquidación"
regla = "no admiten devolución"
seccion = "productos que no se pueden devolver"
print(f"Texto original de la política: {len(texto_politica)} caracteres")
print(f"  Estrategia A (splitter genérico, 512): {len(chunks_fijos)} chunks")
print(f"  Estrategia B (por sección):            {len(chunks_seccion)} chunks")

print(f"\nDonde cae el término '{termino}' (las ofertas de venta final):")
for nombre, chunks in [("A (fijo)", chunks_fijos), ("B (secciones)", chunks_seccion)]:
    coincidencias = buscar(chunks, termino)
    print(f"\n  Estrategia {nombre}: {len(coincidencias)} chunk(s) contienen '{termino}'")
    for ch in coincidencias:
        empieza = ch[:70].replace("\n", " / ")
        tiene_regla = regla in ch.lower()
        tiene_etiqueta = seccion in ch.lower()
        if tiene_regla:
            veredicto = ("la regla explícita 'no admiten devolución' va EN este "
                         "fragmento: el LLM puede citarla.")
        elif tiene_etiqueta:
            veredicto = ("la frase literal no va en este fragmento, pero la etiqueta "
                         "de la sección sí: el contexto de la regla viaja con el "
                         "fragmento, que es lo que el LLM necesita para no inventar.")
        else:
            veredicto = ("este fragmento llega al LLM con el tema ('liquidación') "
                         "pero SIN la regla que lo gobierna ni el encabezado de su "
                         "sección: el LLM puede responder mal a una consulta real.")
        print(f"    - empieza con: '...{empieza}'")
        print(f"    - lectura: {veredicto}")

print("\nLectura del resultado:")
print("Con el splitter genérico (A), la lista de la sección se corta por el techo")
print("de caracteres y el ítem de 'liquidación' termina en un fragmento sin la")
print("regla que lo gobierna ni el encabezado de su sección: el recuperador puede")
print("traer un contexto con el tema pero sin la norma, y el LLM responde mal a")
print("una consulta real sobre las ofertas de venta final.")
print("Con la segmentación por sección (B), aunque esta sección supere el techo y")
print("se corte en dos fragmentos, los dos llevan al inicio la etiqueta con el")
print("nombre de la sección ('Productos que NO se pueden devolver'): el contexto")
print("de la regla viaja completo con cada fragmento. Es la combinación de")
print("chunking por unidad semántica mas etiquetas por chunk la que sostiene")
print("esta propiedad, no el techo de tamaño por sí solo.")


## 5. El LLM y la plantilla de prompts (herencia del Taller Práctico #1)

El LLM generador es el mismo del Taller Práctico #1 (Gemini en su capa gratuita, vía endpoint compatible con OpenAI) y la misma filosofía de plantilla, con el patrón **Rol -> Contexto -> Instrucciones -> Restricciones** (módulo `src/prompts.py`):

- **ROL** (system prompt): agente de servicio al cliente de EcoMarket; solo el bloque de contexto como fuente de verdad; tono cercano; escalar cuando no sepan.
- **CONTEXTO:** los chunks recuperados por el RAG (en el Taller #1 era el JSON del pedido o la política; aquí, los fragmentos que la similitud encontró, cada uno etiquetado con su tipo de documento).
- **INSTRUCCIONES:** responder citando los datos tal como aparecen; si el contexto no alcanza, decirlo con honestidad y ofrecer escalar.
- **RESTRICCIONES:** nada de datos inventados, nada de conocimiento externo al contexto, no revelar detalles técnicos internos.

La plantilla se generaliza respecto al Taller #1: en vez de dos plantillas específicas (pedido y devolución), una sola plantilla RAG atiende cualquier consulta, porque es el *recuperador* el que decide qué material entra al contexto.


In [ ]:
import prompts

# Construye el LLM: Gemini vía endpoint compatible con OpenAI, temperatura
# baja (0.1) porque las cifras y estados citados deben ser fieles al contexto
llm = rag.llamar_llm(os.environ["GEMINI_API_KEY"])

print("System prompt (ROL), primer tramo:")
print(prompts.SYSTEM_AGENTE_ECOMARKET_RAG[:300], "...")

print("\nPlantilla RAG (CONTEXTO + INSTRUCCIONES), primer tramo:")
print(prompts.PROMPT_RAG[:430], "...")


## 6. El asistente completo

La función `rag.responder(...)` integra el pipeline completo:

```
consulta -> recuperar (top-k con distancias)
         -> ¿algo supera el umbral?
              SÍ  -> llenar plantilla del Taller #1 con los chunks -> Gemini -> respuesta
              NO  -> mensaje determinista de honestidad + escalamiento (sin llamar al LLM)
```

El mensaje "no tengo las herramientas" es **determinista** (no lo genera el LLM): es la forma más robusta de garantiza que el asistente nunca improvise frente a solicitudes fuera de su alcance — exactamente el comportamiento exigido por el enunciado ("haciéndole saber al usuario que no tiene las herramientas").


In [ ]:
def preguntar(consulta, k=rag.K_CHUNKS, umbral=rag.UMBRAL_DISTANCIA):
    """Ejecuta el flujo RAG completo para una consulta e imprime el resultado con sus fuentes."""
    print("=" * 76)
    print("CONSULTA:", consulta)
    print("=" * 76)
    respuesta = rag.responder(consulta, vectorstore, llm, k=k, umbral=umbral)
    print()
    print("RESPUESTA DEL ASISTENTE:")
    print(respuesta)
    print()
    return respuesta

print("Función preguntar lista: los casos de prueba de la siguiente sección la usan")


## 7. Casos de prueba (evidencias de la Fase 3)

Cinco casos cubren los comportamientos que evalúa la rúbrica: fidelidad a los datos de la base de conocimiento, síntesis entre documentos, consultas por pedido (con enrutamiento), consultas de catálogo y **solicitud fuera de alcance** (escalamiento honesto). Se usan preguntas directas del tipo que haría un cliente.

| Caso | Consulta | Comportamiento esperado |
|---|---|---|
| 1 | Devolver el café orgánico | Cita la política: los alimentos no se devuelven, con empatía y la excepción por defecto (fidelidad a la política) |
| 2 | ¿Cuánto tarda el reembolso? | Cita la FAQ: 5 días hábiles desde la llegada del producto |
| 3 | Estado del pedido ECO-10003 (retardado) | Disculpa + motivo real (interrupción logística) + nueva fecha + enlace (recuperación enrutada a tipos de pedido) |
| 4 | Precio del kit de compostaje | Cita el inventario: $58.000 COP, disponibilidad |
| 5 | "¿Cuál es el clima en Bogotá?" | Ningún chunk supera el umbral: mensaje determinista de "no tengo las herramientas" (sin generar con el LLM) |

**Nota sobre la capa gratuita:** el modelo `gemini-3.6-flash` admite solo 5 peticiones por minuto en el nivel gratuito de la API de Gemini. Por eso cada celda de esta sección cierra con una pausa de 20 segundos después de la consulta al LLM: la corrida completa se mantiene dentro del límite. Es una limitación del plan gratuito, no del diseño (en producción el contrato del proveedor define el ritmo); el notebook la documenta porque el prototipo debe ejecutarse de manera repetible.


In [ ]:
# La pausa de 20 segundos mantiene la corrida dentro del límite de
# peticiones por minuto de la capa gratuita de la API de Gemini (5/min)
import time

resp_caso1 = preguntar("Quiero devolver el café orgánico molido que compré. ¿Puedo?")
time.sleep(20)


In [ ]:
resp_caso2 = preguntar("¿Cuánto tarda el reembolso de una devolución?")
time.sleep(20)


In [ ]:
resp_caso3 = preguntar("¿Qué pasa con mi pedido ECO-10003? El envío lleva muchos días.")
time.sleep(20)


In [ ]:
resp_caso4 = preguntar("¿En qué precio está el kit de compostaje de cocina y está disponible?")
time.sleep(20)


In [ ]:
# Fuera de dominio: ningún documento de EcoMarket describe el clima; el sistema
# NO consulta al LLM con contexto pobre sino que comunica el límite de alcance
resp_caso5 = preguntar("¿Cuál es el clima en Bogotá para el fin de semana?")
time.sleep(20)


#### Lectura de los resultados de los casos 1-5

- **Casos 1, 2 y 4:** el bloque de fuentes impreso antes de cada respuesta muestra qué chunks recuperó y con qué distancias. La respuesta debe citar cifras exactas del corpus (plazo de 30 días, 5 días hábiles, $58.000) — la fidelidad a los datos es el efecto del contexto y del patrón de restricciones.
- **Caso 3:** la lista de fuentes contiene solo documentos de tipo `pedido` (el enrutamiento por número de seguimiento evitó ruido), y el motivo del retrazo citado debe ser el real del registro (interrupción logística Cali-Medellín), no una justificación genérica: esa diferencia se evidenció en el Taller Práctico #1 comparando con y sin contexto.
- **Caso 5:** el texto impreso después de "RESPUESTA DEL ASISTENTE" es el mensaje determinista (no hay generación del LLM): el sistema imprimió el bloque de fuentes, detectó que ninguna distancia superó el umbral y devolvió el mensaje de escalamiento. Este es el "haciéndole saber que no tiene las herramientas" del enunciado.


## 8. Experimento de control — la misma consulta sin RAG

La rúbrica de la Fase 3 premia que el sistema permita "modificar el código o el prompt de forma exitosa para observar cambios en el comportamiento del modelo". Este experimento apaga deliberadamente el sistema RAG y envía al LLM solamente la consulta del Caso 1 (con una instrucción mínima y sin bloque de contexto), replicando el control `--basico` del Taller Práctico #1.

La variable única que se modifica es el **contexto recuperado**. Lo que se observa: sin la base de conocimiento, el modelo puede a) decir vagamente que no sabe (mejor caso) o b) *inventar* una política de devoluciones plausible para la tienda ficticia EcoMarket, sin citar el plazo de 30 días ni la excepción por defecto de fábrica — es decir, alucinación con apariencia de respuesta oficial, el riesgo central que el RAG ataca.


In [ ]:
resp_control = rag.responder_sin_rag("Quiero devolver el café orgánico molido que compré. ¿Puedo?", llm)

print("=" * 76)
print("MISMA CONSULTA, SIN RAG (sin contexto recuperado, sin restricciones)")
print("=" * 76)
print(resp_control)
time.sleep(20)


#### Conclusión del experimento

Al comparar la respuesta de `resp_caso1` (con RAG) con `resp_control` (sin RAG) se observa el cambio de comportamiento al agregar y quitar el sistema RAG:

- **Con RAG:** la respuesta se ancla a la política de la empresa (categoría "alimentos y bebidas" no devolvible, excepción por defecto de fábrica, plazo de 7 días para defectos). Nada inventado.
- **Sin RAG:** el modelo opera con su conocimiento general: o pide disculpas genéricas sin comprometer datos, o fabrica condiciones que en EcoMarket no existen. En ambos casos la respuesta no es útil para el negocio: la versión sin RAG no puede citar el plazo de 30 días ni el proceso de etiqueta de retorno, porque no los conoce.

Con esto se demuestra que el cambio observado (añadir/quitar RAG) gobierna el comportamiento del sistema, y que la calidad del contexto — no solo del prompt — es el factor que sostiene la precisión del asistente.


## 9. Limitaciones y suposiciones

El detalle completo está en `docs/fase3-arquitectura-y-limitaciones.md`, sección 3. En síntesis:

1. **Corpus simulado y pequeño** (4 documentos, menos de 100 chunks): la validación aplica al prototipo, no a producción.
2. **Sin conexión en vivo a un backend:** pedidos e inventario son estáticos; la frescura se resolvería con sincronización/reindexación programada contra el ERP.
3. **Umbral de similitud calibrado empíricamente** con los casos de prueba; para otro corpus habría que recalibrarlo.
4. **La calidad de los embeddings open-source es suficiente para este volumen**; con corpora miles de veces mayores y ambigüedad alta se reabriría la discusión sobre modelos propietarios o híbrids (decisión documentada en la Fase 1).
5. **Consultas monovuelta:** no hay memoria de diálogo multivuelta; añadirla es una mejora natural en la línea de agentes de la unidad 3.
6. **Recursos:** el equipo ejecuta en Colab (sin GPU y con disco limitado en los equipos locales), por lo que la solución se diseñó 100% gratuita: sin descargas de modelos grandes locales ni servicios de pago.

## 10. Escalabilidad y riesgos éticos (breve reflexión, requerida en la Semana 4)

**Escalabilidad.** El diseño escala del prototipo a piloto sin cambiar de componentes (ChromaDB soporta millones de vectores). El volumen de la base de conocimiento no cambia la arquitectura; lo que cambia la arquitectura son: la necesidad de búsqueda híbrida cuando la similitud pura introduzca ruido (allí entrarían Weaviate self-hosted o Pinecone, re-evaluando costo como en la Fase 1), y la sincronización continua con el ERP para que los estados de pedidos no queden desactualizados — un RAG con datos viejos responde con más confianza que uno sin RAG y con menos exactitud.

**Riesgos éticos** (continuando la Fase 2 del Taller Práctico #1):

- **Alucinación residual:** el RAG y el umbral la reducen drásticamente (el experimento de la sección 8 muestra el contraste), no la eliminan; el modelo puede reformular mal lo recuperado. Mitigación: restricciones del prompt, casos de prueba y escalamiento a humanos.
- **Privacidad:** los chunks de pedidos incluyen nombre del cliente; en producción, la consulta a chunks de pedido se haría autenticada por el cliente, y los embeddings ya se calculan dentro del entorno (nada viaja a terceros por vectorizar). Tratamiento conforme a la Ley 1581 de 2012.
- **Transparencia:** el cliente sabe que habla con un asistente virtual; cuando el sistema no puede responder, lo comunica en lugar de simular una consultoría experta, y ofrece un canal humano real (FAQ-11).
- **Impacto laboral:** el sistema absorbe consultas repetitivas (el 80% del planteamiento del caso) y deriva a los asesores humanos los casos de mayor valor: automatización que empodera al equipo de soporte en lugar de reemplazarlo.
- **Sesgo:** la base de conocimiento define qué puede responder el asistente; una curaduría incompleta desbalancea el servicio (por ejemplo, si faltaran canales accesibles para personas con discapacidad visual). El diseño de la Fase 2 es también una decisión ética.


## 11. Conclusiones

El taller quedó completo en sus tres fases:

1. **Fase 1 (componentes):** modelo de embeddings open-source multilingüe y ChromaDB, seleccionados y justificados contra las alternativas propietarias y gestionadas con criterios de rendimiento en español, costo, privacidad, escalabilidad y carga operativa — y ligados a las necesidades específicas de EcoMarket (corpus pequeño, datos de clientes bajo control, presupuesto cero).
2. **Fase 2 (base de conocimiento):** cuatro tipos de documentos identificados con su rol en soporte; segmentación por unidad semántica natural de cada tipo (secciones, registros, pares Q-A); indexación con espacio coseno y persistencia, con la decisión calibrada al límite de entrada del modelo de embeddings.
3. **Fase 3 (integración):** asistente funcional que atiende cualquier consulta sobre la empresa con recuperación semántica, verifica la pertinencia de lo recuperado (umbral) y comunica con honestidad cuando no tiene las herramientas — el sistema RAG está montado sobre el patrón de prompts del Taller Práctico #1, y el experimento de control demuestra el cambio de comportamiento al agregar/quitar el RAG.

El resultado materializa de forma concreta la unidad 3: un flujo de automatización con un agente (el asistente), una herramienta de ampliación de conocimiento (el recuperador RAG) y una reflexión en ejecución (el umbral de pertinencia y el escalamiento honesto).
